### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [1]:
import os 
from dotenv import load_dotenv 
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")


python-dotenv could not parse statement starting at line 3


In [2]:
from langchain.chat_models import init_chat_model

model=init_chat_model(model="groq:qwen/qwen3-32b")

response=model.invoke("What is ai ")
response 

AIMessage(content='<think>\nOkay, so the user is asking "What is AI?" I need to explain AI in a clear and concise way. Let me start by breaking down the term. AI stands for Artificial Intelligence. But what does that actually mean? I should define it first.\n\nArtificial Intelligence refers to the simulation of human intelligence in machines that are programmed to think like humans and mimic their actions. The key here is that it\'s about machines performing tasks that typically require human intelligence. But I need to give examples to make it concrete. Things like problem-solving, learning, understanding natural language, recognizing patterns, maybe even decision-making.\n\nWait, there are different types of AI, right? I should mention that. There\'s narrow AI, which is focused on specific tasks, like Siri or recommendation systems. Then there\'s general AI, which is more about machines being able to handle any intellectual task a human can, but that\'s more theoretical and not yet a

In [ ]:
!pip install langchain.tools

In [ ]:
from langchain.tools import tool

@tool # tools decorator
def get_weather(location:str)->str: # input will be location in the form of string 
    """Get the weather at  a location """
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])  # one way to bind tools with model...now its an agent 

# # Second method to bind tools with model 
# from langchain.agents import create_agent

# model_with_tools=create_agent(
#     model=model,
#     tools=[get_weather]
# )    

In [ ]:
response = model_with_tools.invoke("What is the weather in Boston")
print(response)


    

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Boston is the location they mentioned. I need to call that function with "Boston" as the argument. I\'ll make sure the JSON is correctly formatted with the location parameter. No other parameters are needed since the function only requires location. Let me double-check the syntax to avoid any errors.\n', 'tool_calls': [{'id': '1yf0zy2gk', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 114, 'prompt_tokens': 153, 'total_tokens': 267, 'completion_time': 0.188963384, 'completion_tokens_details': {'reasoning_tokens': 90}, 'prompt_time': 0.00605062, 'prompt_tokens_details': None, 'queue_time': 0.054086926, 'total_time': 0.195014004}, 'model_name': 'qwen/qwen3-32b', 'system_fingerpr

In [13]:
for tool_call in response.tool_calls:
    # view tool calls made by the model 
    print(f"Total : {tool_call['name']}")
    print(f"Args  : {tool_call['args']}")

Total : get_weather
Args  : {'location': 'Boston'}


Tool Execution Loops 

In [14]:
# Step 1: Model generates tool calls
messages = [{"role":"user","content":"What is the weather in Boston"}]
ai_msg=model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2  : Execute tools and collect results 
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments 
    tool_result=get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3 : Pass results back to model for final response 
final_response=model_with_tools.invoke(messages)
print(final_response.text)



The weather in Boston is sunny. A perfect day to enjoy outdoor activities! ☀️


In [15]:
messages

[{'role': 'user', 'content': 'What is the weather in Boston'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user specified Boston, I need to call that function with "Boston" as the location. I\'ll make sure to format the tool call correctly within the XML tags.\n', 'tool_calls': [{'id': '1ddc9sg0p', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 91, 'prompt_tokens': 153, 'total_tokens': 244, 'completion_time': 0.128696452, 'completion_tokens_details': {'reasoning_tokens': 67}, 'prompt_time': 0.006141266, 'prompt_tokens_details': None, 'queue_time': 0.160213363, 'total_time': 0.134837718}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand', 'finish_re